# Sweden Personal EV Electricity Load Curve

This notebook generates an 8,760-hour annual load profile for personal EV charging in Sweden.

The profile combines three components:
1. **Base 24-hour charging pattern** (from `personbilar_effektprofil.csv`)
2. **Monthly seasonal multipliers** (winter 35% higher than summer)
3. **Day-of-week multipliers** (weekdays 16% higher than weekends)

See `README.md` for detailed methodology and sources.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

# Configuration
PROFILES_PATH = Path.cwd()
OUTPUT_PATH = PROFILES_PATH.parent  # Export to parent directory (load_profiles/)
DATA_YEAR = 2024

def is_leap_year(year: int) -> bool:
    return year % 4 == 0 and (year % 100 != 0 or year % 400 == 0)

HOURS_IN_YEAR = 8784 if is_leap_year(DATA_YEAR) else 8760

# Monthly seasonal multipliers (from README - Nordic EV research)
# Winter consumption 25-35% higher than summer due to battery heating, cabin heating, etc.
SEASONAL_MULTIPLIERS = {
    1: 1.28,   # January - Peak winter
    2: 1.28,   # February
    3: 1.18,   # March
    4: 1.10,   # April
    5: 0.95,   # May
    6: 0.95,   # June - Lowest
    7: 1.05,   # July (AC usage)
    8: 1.05,   # August (AC usage)
    9: 1.08,   # September
    10: 1.12,  # October
    11: 1.18,  # November
    12: 1.25,  # December
}

# Day-of-week multipliers (Monday=0, Sunday=6)
# Based on Swedish National Travel Survey (RVU Sweden)
WEEKDAY_MULTIPLIERS = {
    0: 1.03,  # Monday
    1: 1.03,  # Tuesday
    2: 1.03,  # Wednesday
    3: 1.03,  # Thursday
    4: 1.00,  # Friday
    5: 0.90,  # Saturday
    6: 0.87,  # Sunday
}

print(f"Year: {DATA_YEAR} ({'leap' if is_leap_year(DATA_YEAR) else 'normal'})")
print(f"Hours in year: {HOURS_IN_YEAR}")
print(f"Seasonal range: {min(SEASONAL_MULTIPLIERS.values()):.2f} - {max(SEASONAL_MULTIPLIERS.values()):.2f}")
print(f"Weekday range: {min(WEEKDAY_MULTIPLIERS.values()):.2f} - {max(WEEKDAY_MULTIPLIERS.values()):.2f}")

## 1. Load Base 24-Hour Profile

The base charging profile represents typical daily charging patterns for Swedish residential EV users.
- Peak at midnight (hour 0): home charging after evening commute
- Trough at 9 AM (hour 9): minimal charging during work hours

In [ ]:
# Load the base 24-hour profile
df_base = pd.read_csv(PROFILES_PATH / 'personbilar_effektprofil.csv')
base_24h = df_base['power'].values

print(f"Base profile: {len(base_24h)} hours")
print(f"Sum: {base_24h.sum():.6f} (should be 1.0)")
print(f"Peak hour: {base_24h.argmax()} (value: {base_24h.max():.4f})")
print(f"Trough hour: {base_24h.argmin()} (value: {base_24h.min():.4f})")
print(f"Peak/Trough ratio: {base_24h.max() / base_24h.min():.1f}x")

## 2. Visualize Base Profile and Multipliers

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Base 24-hour profile
ax1 = axes[0]
ax1.bar(range(24), base_24h, color='steelblue', alpha=0.7)
ax1.axhline(y=1/24, color='red', linestyle='--', alpha=0.5, label='Flat')
ax1.set_xlabel('Hour of Day')
ax1.set_ylabel('Relative Power')
ax1.set_title('Base 24-Hour Profile')
ax1.set_xticks(range(0, 24, 3))

# Monthly seasonal multipliers
ax2 = axes[1]
months = list(SEASONAL_MULTIPLIERS.keys())
values = list(SEASONAL_MULTIPLIERS.values())
colors = ['#1f77b4' if v > 1.0 else '#ff7f0e' for v in values]
ax2.bar(months, values, color=colors, alpha=0.7)
ax2.axhline(y=1.0, color='black', linestyle='--', alpha=0.5)
ax2.set_xlabel('Month')
ax2.set_ylabel('Multiplier')
ax2.set_title('Seasonal Multipliers')
ax2.set_xticks(months)

# Weekday multipliers
ax3 = axes[2]
days = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
dow_values = [WEEKDAY_MULTIPLIERS[i] for i in range(7)]
colors = ['#1f77b4' if v >= 1.0 else '#ff7f0e' for v in dow_values]
ax3.bar(days, dow_values, color=colors, alpha=0.7)
ax3.axhline(y=1.0, color='black', linestyle='--', alpha=0.5)
ax3.set_ylabel('Multiplier')
ax3.set_title('Day-of-Week Multipliers')

plt.tight_layout()
plt.show()

In [ ]:
## 3. Generate Annual Profile

For each hour of the year:
```
raw_value = base_24h[hour] × seasonal[month] × weekday[day_of_week]
```

Then normalize so all values sum to 1.0.

def generate_ev_annual_profile(base_24h: np.ndarray, year: int) -> pd.DataFrame:
    """
    Generate annual EV charging profile with seasonal and weekday variation.
    
    For each hour: raw = base_24h[hour] × seasonal[month] × weekday[dow]
    Then normalize to sum = 1.0
    """
    hours = 8784 if is_leap_year(year) else 8760
    timestamps = pd.date_range(f"{year}-01-01", periods=hours, freq="h")
    
    raw_values = np.zeros(hours)
    
    for i, ts in enumerate(timestamps):
        hour = ts.hour
        month = ts.month
        dow = ts.dayofweek  # Monday=0, Sunday=6
        
        raw_values[i] = base_24h[hour] * SEASONAL_MULTIPLIERS[month] * WEEKDAY_MULTIPLIERS[dow]
    
    # Normalize to sum = 1.0
    normalized = raw_values / raw_values.sum()
    
    return pd.DataFrame({
        'timestamp': timestamps,
        'hour': range(hours),
        'value': normalized
    })

# Generate the profile
df_annual = generate_ev_annual_profile(base_24h, DATA_YEAR)

print(f"Generated annual profile: {len(df_annual)} hours")
print(f"Sum: {df_annual['value'].sum():.10f}")
print(f"Min value: {df_annual['value'].min():.8f}")
print(f"Max value: {df_annual['value'].max():.8f}")

In [ ]:
## 4. Validate Profile

Check all validation criteria from the README:

In [ ]:
def validate_profile(df: pd.DataFrame) -> bool:
    """Validate against README criteria."""
    checks = []
    
    # 1. Sum equals 1.0
    total = df['value'].sum()
    checks.append(('Sum = 1.0', abs(total - 1.0) < 1e-9, f"{total:.10f}"))
    
    # 2. All positive
    all_positive = (df['value'] > 0).all()
    checks.append(('All positive', all_positive, f"min={df['value'].min():.2e}"))
    
    # 3. January > June
    jan = df[df['timestamp'].dt.month == 1]['value'].sum()
    jun = df[df['timestamp'].dt.month == 6]['value'].sum()
    ratio = jan / jun
    checks.append(('January > June', jan > jun, f"ratio={ratio:.2f}"))
    
    # 4. Weekdays > Weekends
    weekday = df[df['timestamp'].dt.dayofweek < 5]['value'].sum()
    weekend = df[df['timestamp'].dt.dayofweek >= 5]['value'].sum()
    ratio_wd = weekday / weekend
    checks.append(('Weekdays > Weekends', weekday > weekend, f"ratio={ratio_wd:.2f}"))
    
    # 5. Peak at hour 0
    hourly = df.groupby(df['timestamp'].dt.hour)['value'].sum()
    peak_hour = hourly.idxmax()
    checks.append(('Peak at hour 0', peak_hour == 0, f"peak={peak_hour}"))
    
    # 6. Trough at hour 9
    trough_hour = hourly.idxmin()
    checks.append(('Trough at hour 9', trough_hour == 9, f"trough={trough_hour}"))
    
    print("Validation Results:")
    print("-" * 50)
    for name, passed, detail in checks:
        status = '✓' if passed else '✗'
        print(f"  {status} {name:<25} ({detail})")
    
    all_passed = all(passed for _, passed, _ in checks)
    print("-" * 50)
    print(f"  {'✓ ALL PASSED' if all_passed else '✗ FAILED'}")
    
    return all_passed

# Run validation
validate_profile(df_annual)

In [ ]:
## 5. Visualize Results

# Monthly totals
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Monthly totals
ax1 = axes[0, 0]
monthly = df_annual.groupby(df_annual['timestamp'].dt.month)['value'].sum()
colors = ['#1f77b4' if m in [1, 2, 11, 12] else '#ff7f0e' if m in [5, 6] else '#2ca02c' for m in monthly.index]
ax1.bar(monthly.index, monthly.values, color=colors, alpha=0.7)
ax1.set_xlabel('Month')
ax1.set_ylabel('Total Load (normalized)')
ax1.set_title('Monthly Energy Totals')
ax1.set_xticks(range(1, 13))

# Day-of-week totals
ax2 = axes[0, 1]
daily = df_annual.groupby(df_annual['timestamp'].dt.dayofweek)['value'].sum()
days = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
colors = ['#1f77b4' if i < 5 else '#ff7f0e' for i in range(7)]
ax2.bar(days, daily.values, color=colors, alpha=0.7)
ax2.set_ylabel('Total Load (normalized)')
ax2.set_title('Day-of-Week Totals')

# Winter week (January)
ax3 = axes[1, 0]
winter_week = df_annual[(df_annual['timestamp'].dt.month == 1) & 
                         (df_annual['timestamp'].dt.day <= 7)]
ax3.plot(range(len(winter_week)), winter_week['value'].values, color='#1f77b4', linewidth=0.8)
for d in range(8):
    ax3.axvline(x=d*24, color='gray', linestyle=':', alpha=0.3)
ax3.set_xlabel('Hour')
ax3.set_ylabel('Normalized Value')
ax3.set_title('Sample Winter Week (Jan 1-7)')

# Summer week (June)
ax4 = axes[1, 1]
summer_week = df_annual[(df_annual['timestamp'].dt.month == 6) & 
                         (df_annual['timestamp'].dt.day <= 7)]
ax4.plot(range(len(summer_week)), summer_week['value'].values, color='#ff7f0e', linewidth=0.8)
for d in range(8):
    ax4.axvline(x=d*24, color='gray', linestyle=':', alpha=0.3)
ax4.set_xlabel('Hour')
ax4.set_ylabel('Normalized Value')
ax4.set_title('Sample Summer Week (Jun 1-7)')

plt.tight_layout()
plt.show()

# Print comparison
print(f"\nWinter/Summer comparison:")
print(f"  January total:  {monthly[1]:.4f}")
print(f"  June total:     {monthly[6]:.4f}")
print(f"  Ratio:          {monthly[1]/monthly[6]:.2f}x")

print(f"\nWeekday/Weekend comparison:")
weekday_avg = daily[:5].mean()
weekend_avg = daily[5:].mean()
print(f"  Weekday avg:    {weekday_avg:.5f}")
print(f"  Weekend avg:    {weekend_avg:.5f}")
print(f"  Ratio:          {weekday_avg/weekend_avg:.2f}x")

In [ ]:
## 6. Export Profile

In [ ]:
# Export to parent directory (load_profiles/)
output_file = OUTPUT_PATH / f'profile_transport_cars_{DATA_YEAR}.csv'

# Export with hour index only (matching other profiles)
df_export = df_annual[['hour', 'value']].copy()
df_export.to_csv(output_file, index=False)

print(f"Exported: {output_file}")
print(f"Rows: {len(df_export)}")
print(f"Columns: {df_export.columns.tolist()}")

# Verify
df_verify = pd.read_csv(output_file)
print(f"\nVerification:")
print(f"  Rows: {len(df_verify)}")
print(f"  Sum: {df_verify['value'].sum():.10f}")

# List all profile files
print(f"\nProfile files in {OUTPUT_PATH.name}/:")
for f in sorted(OUTPUT_PATH.glob('profile_*.csv')):
    print(f"  {f.name}")

## 7. Export Patterns for Multi-Year Extension

Export the underlying patterns (hourly, weekday, monthly) as JSON for use with `extend_from_pattern()`.

In [ ]:
import json

# Prepare patterns for JSON export
# Convert from our multiplier format to the extend_from_pattern() format

# 1. Hourly pattern (already normalized, sum=1.0)
hourly_pattern = base_24h.tolist()
print(f"Hourly pattern: {len(hourly_pattern)} values, sum={sum(hourly_pattern):.4f}")

# 2. Weekday multipliers (7 values, Mon-Sun, relative to mean)
weekday_values = [WEEKDAY_MULTIPLIERS[i] for i in range(7)]
weekday_mean = sum(weekday_values) / 7
weekday_mult_normalized = [v / weekday_mean for v in weekday_values]
print(f"Weekday multipliers: {[round(v, 4) for v in weekday_mult_normalized]}")
print(f"  Sum: {sum(weekday_mult_normalized):.2f} (should be ~7.0)")

# 3. Monthly multipliers (12 values, relative to mean)
monthly_values = [SEASONAL_MULTIPLIERS[m] for m in range(1, 13)]
monthly_mean = sum(monthly_values) / 12
monthly_mult_normalized = [v / monthly_mean for v in monthly_values]
print(f"Monthly multipliers: {[round(v, 4) for v in monthly_mult_normalized]}")
print(f"  Sum: {sum(monthly_mult_normalized):.2f} (should be ~12.0)")

# Create patterns dict
patterns = {
    'hourly': hourly_pattern,
    'weekday': weekday_mult_normalized,  # 7 values (Mon-Sun)
    'monthly': monthly_mult_normalized,
    'source_year': DATA_YEAR,
    'description': 'Personal EV charging patterns for Sweden based on Nordic research',
    'data_sources': [
        'personbilar_effektprofil.csv (base 24h pattern)',
        'Nordic EV charging behavior research (seasonal)',
        'Swedish National Travel Survey - RVU (weekday)'
    ]
}

# Export to JSON
pattern_file = OUTPUT_PATH / 'profile_transport_cars_patterns.json'
with open(pattern_file, 'w') as f:
    json.dump(patterns, f, indent=2)

print(f"\nPatterns exported to: {pattern_file}")
print(f"Use with extend_from_pattern() for multi-year timeseries generation.")